# Metrics + Answer Normalization

Defines shared normalization and evaluation metrics for Kvasir-VQA x1.
Exports reusable utilities in `utils/metrics.py`.


In [ ]:
import os
import sys
from pathlib import Path

import pandas as pd

# Add dataset root to sys.path so `utils` is importable

def find_kvasir_x1_root() -> Path:
    env_root = os.environ.get("KVASIR_VQA_X1_ROOT")
    if env_root:
        p = Path(env_root).expanduser().resolve()
        if (p / "0_dataset_prep").exists():
            return p
        raise RuntimeError(f"KVASIR_VQA_X1_ROOT set but missing 0_dataset_prep: {p}")

    p = Path.cwd().resolve()
    for _ in range(6):
        if (p / "Prototyping_reformat" / "DatasetAnalysis" / "Kvasir_VQA_x1").exists():
            return (p / "Prototyping_reformat" / "DatasetAnalysis" / "Kvasir_VQA_x1").resolve()
        if (p / "0_dataset_prep").exists() and (p / "1_dataset_analysis").exists():
            return p
        p = p.parent
    raise RuntimeError("Could not locate Kvasir_VQA_x1 root. Set KVASIR_VQA_X1_ROOT.")

ROOT = find_kvasir_x1_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from utils.metrics import normalize_answer, exact_match, token_f1, anls, compute_metrics

print("ROOT:", ROOT)


In [ ]:
# Quick sanity check for normalization
examples = [
    "  The Polyp! ",
    "No polypoid lesions identified.",
    "Z-line",
    "   evidence of   oesophagitis  ",
]
for e in examples:
    print(e, "->", normalize_answer(e))


In [ ]:
# Metric examples
preds = ["yes", "no polyp", "evidence of oesophagitis"]
golds = ["yes", "no polypoid lesions identified", "oesophagitis"]

print("EM:", [exact_match(p, g) for p, g in zip(preds, golds)])
print("Token-F1:", [token_f1(p, g) for p, g in zip(preds, golds)])
print("ANLS:", [anls(p, g) for p, g in zip(preds, golds)])

print("Summary:", compute_metrics(preds, golds))


In [ ]:
# Optional: evaluate a predictions file
# (expects columns: pred, answer)
PRED_PATH = None  # set to a CSV/JSONL if you want a quick test

if PRED_PATH:
    df = pd.read_csv(PRED_PATH)
    metrics = compute_metrics(df["pred"].tolist(), df["answer"].tolist())
    print(metrics)
